# Perform feature selection on normalized data

## Import libraries

In [1]:
import os
import pathlib
import pprint

import pandas as pd
from pycytominer import feature_select
from pycytominer.cyto_utils import output
from timelapse_utils.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from timelapse_utils.profiling_utils.sc_extraction_utils import add_single_cell_count_df

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm

## Set paths and variables

In [2]:
# load in platemap file as a pandas dataframe
platemap_path = pathlib.Path(
    f"{root_dir}/Wave2_data/0.download_data/platemap/platemap.csv"
).resolve()

image_base_dir = bandicoot_check(
    bandicoot_mount_path=pathlib.Path(f"{os.path.expanduser('~')}/mnt/bandicoot/"),
    root_dir=root_dir,
)
image_base_dir = pathlib.Path(
    f"{image_base_dir}/live_cell_timelapse_pyroptosis_project_data/processed_data/"
).resolve(strict=True)
normalized_profiles_path = pathlib.Path(
    f"{image_base_dir}/7.normalized_profiles/normalized_profiles_wave2.parquet"
).resolve(strict=True)

feature_selected_profiles_path = pathlib.Path(
    f"{image_base_dir}/8.feature_selected_profiles/feature_selected_profiles_wave2.parquet"
).resolve()
feature_selected_profiles_path.parent.mkdir(exist_ok=True)

## Define dict of paths

## Perform feature selection

In [3]:
# define operations to be performed on the data
# list of operations for feature select function to use on input profile
feature_select_ops = [
    "variance_threshold",
    "blocklist",
    "drop_na_columns",
    "correlation_threshold",
]

This last cell does not get run due to memory constraints. 
It is run on an HPC cluster with more memory available.

In [4]:
# read in the annotated file
normalized_df = pd.read_parquet(normalized_profiles_path)
metadata_cols = [x for x in normalized_df.columns if x.startswith("Metadata_")]
normalized_features_df = normalized_df.drop(metadata_cols, axis="columns")
# perform feature selection with the operations specified
feature_select_df = feature_select(
    normalized_features_df,
    operation=feature_select_ops,
)

# add metadata columns back to the feature selected df
feature_select_df = pd.concat(
    [normalized_df[metadata_cols], feature_select_df], axis="columns"
)
print("Feature selection complete, saving to parquet file!")
# save features selected df as parquet file
output(
    df=feature_select_df,
    output_filename=feature_selected_profiles_path,
    output_type="parquet",
)
# check to see if the shape of the df has changed indicating feature selection occurred
print(feature_select_df.shape)
feature_select_df.head()

Feature selection complete, saving to parquet file!
(4811, 1051)


,Metadata_Well,Metadata_Inducer,Metadata_Inducer_dose,Metadata_Inhibitor,Metadata_Inhibitor_dose,Metadata_Time,Metadata_Well_FOV,Metadata_FOV,Metadata_Well_FOV_Time,Metadata_ImageNumber,...,Nuclei_Texture_DifferenceEntropy_BF_3_02_256,Nuclei_Texture_DifferenceEntropy_BF_3_03_256,Nuclei_Texture_DifferenceVariance_BF_3_00_256,Nuclei_Texture_DifferenceVariance_BF_3_01_256,Nuclei_Texture_DifferenceVariance_BF_3_02_256,Nuclei_Texture_InverseDifferenceMoment_BF_3_01_256,Nuclei_Texture_InverseDifferenceMoment_CL640_3_00_256,Nuclei_Texture_SumEntropy_BF_3_02_256,Nuclei_Texture_SumVariance_BF_3_01_256,Nuclei_Texture_SumVariance_BF_3_03_256
0,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,-0.676610,-0.209997,-0.007872,0.408620,0.846543,1.402981,0.347620,-0.126906,0.528042,1.370239
1,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,-0.540441,-2.320790,1.038346,-0.365552,1.118915,-0.563051,2.034756,-0.892038,-1.657563,0.469979
2,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,-1.275388,-0.589885,0.797691,2.412517,0.943863,1.995294,0.973340,-0.688565,-0.634257,-1.147281
3,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,-3.012013,-1.159804,1.833922,4.426940,3.963932,2.779536,2.409483,-1.362258,-1.229978,-1.838455
4,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,0.711109,1.510042,-1.584158,-1.542460,-1.178280,-2.275320,-0.293348,2.024404,3.392645,3.305974
